In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
df = pd.read_csv(r"C:\Users\cecil\Cript_Anomalies\BTCUSDT_20180101_20260112.csv", delimiter=";", skiprows=0)
display(df)

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class AffineCoupling(nn.Module):
    """
    Affine coupling layer (RealNVP) with binary mask.
    x1 = x * mask, x2 = x * (1-mask)
    s,t are predicted from x1 and applied to x2
    """
    def __init__(self, dim, hidden_dim, mask):
        super().__init__()
        self.dim = dim
        self.register_buffer("mask", mask)  # shape [dim]
        in_dim = int(mask.sum().item())
        out_dim = dim - in_dim
        self.st_net = MLP(in_dim, hidden_dim, out_dim * 2)

    def forward(self, x):
        x1 = x[:, self.mask.bool()]
        x2 = x[:, (~self.mask.bool())]

        st = self.st_net(x1)
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s)  # keep scale stable

        y2 = x2 * torch.exp(s) + t
        y = x.clone()
        y[:, self.mask.bool()] = x1
        y[:, (~self.mask.bool())] = y2

        log_det = s.sum(dim=1)
        return y, log_det

    def inverse(self, y):
        y1 = y[:, self.mask.bool()]
        y2 = y[:, (~self.mask.bool())]

        st = self.st_net(y1)
        s, t = st.chunk(2, dim=1)
        s = torch.tanh(s)

        x2 = (y2 - t) * torch.exp(-s)
        x = y.clone()
        x[:, self.mask.bool()] = y1
        x[:, (~self.mask.bool())] = x2

        log_det = (-s).sum(dim=1)
        return x, log_det


class RealNVP(nn.Module):
    def __init__(self, dim, n_coupling=6, hidden_dim=128):
        super().__init__()
        self.dim = dim
        layers = []

        # Alternating masks
        for i in range(n_coupling):
            if i % 2 == 0:
                mask = torch.tensor([1 if j % 2 == 0 else 0 for j in range(dim)], dtype=torch.float32)
            else:
                mask = torch.tensor([0 if j % 2 == 0 else 1 for j in range(dim)], dtype=torch.float32)
            layers.append(AffineCoupling(dim, hidden_dim, mask))

        self.layers = nn.ModuleList(layers)
        self.base = torch.distributions.MultivariateNormal(
            loc=torch.zeros(dim),
            covariance_matrix=torch.eye(dim)
        )

    def f(self, x):
        # x -> z
        log_det_sum = torch.zeros(x.size(0), device=x.device)
        z = x
        for layer in self.layers:
            z, log_det = layer(z)
            log_det_sum += log_det
        return z, log_det_sum

    def g(self, z):
        # z -> x
        log_det_sum = torch.zeros(z.size(0), device=z.device)
        x = z
        for layer in reversed(self.layers):
            x, log_det = layer.inverse(x)
            log_det_sum += log_det
        return x, log_det_sum

    def log_prob(self, x):
        z, log_det = self.f(x)
        return self.base.log_prob(z) + log_det

In [ ]:
def fit_normalizing_flow(
    df: pd.DataFrame,
    features,
    contamination: float = 0.01,
    n_coupling: int = 6,
    hidden_dim: int = 128,
    epochs: int = 15,
    batch_size: int = 2048,
    lr: float = 1e-3,
    device: str | None = None,
    random_state: int = 42,
):
    """
    Devuelve:
      - df_result con anomaly_flow y flow_score
      - flow_model entrenado
      - scaler
      - threshold usado para marcar anomalías
    """

    torch.manual_seed(random_state)
    np.random.seed(random_state)

    df_model = df.copy()

    X = (
        df_model[features]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )
    idx = X.index

    scaler = StandardScaler()
    Xs = scaler.fit_transform(X.values).astype(np.float32)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    dataset = TensorDataset(torch.from_numpy(Xs))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=False)

    dim = Xs.shape[1]
    flow = RealNVP(dim=dim, n_coupling=n_coupling, hidden_dim=hidden_dim).to(device)

    opt = torch.optim.Adam(flow.parameters(), lr=lr)

    flow.train()
    for ep in range(1, epochs + 1):
        losses = []
        for (xb,) in loader:
            xb = xb.to(device)
            loss = -flow.log_prob(xb).mean()  # NLL
            opt.zero_grad()
            loss.backward()
            opt.step()
            losses.append(loss.item())

        # print opcional (puedes comentar si no quieres output)
        print(f"Epoch {ep}/{epochs} - NLL: {np.mean(losses):.4f}")

    # Scoring (inference)
    flow.eval()
    with torch.no_grad():
        x_all = torch.from_numpy(Xs).to(device)
        logp = flow.log_prob(x_all).cpu().numpy()
        score = (-logp)  # mayor = más anómalo

    # Umbral por percentil (contamination)
    threshold = np.quantile(score, 1 - contamination)

    df_model["flow_score"] = np.nan
    df_model["anomaly_flow"] = np.nan
    df_model.loc[idx, "flow_score"] = score
    df_model.loc[idx, "anomaly_flow"] = score >= threshold

    return df_model, flow, scaler, float(threshold)

In [ ]:
def compute_flow_metrics(df: pd.DataFrame):
    df_eval = df.dropna(subset=["anomaly_flow"])

    total = len(df_eval)
    anomalies = df_eval["anomaly_flow"].sum()
    rate = anomalies / total if total else 0

    metrics = {
        "total_points": total,
        "total_anomalies": int(anomalies),
        "anomaly_rate": float(rate),
        "score_mean": float(df_eval["flow_score"].mean()),
        "score_std": float(df_eval["flow_score"].std()),
        "score_p95": float(df_eval["flow_score"].quantile(0.95)),
        "score_p99": float(df_eval["flow_score"].quantile(0.99)),
        "score_max": float(df_eval["flow_score"].max()),
    }

    if "anomaly_simple" in df.columns:
        overlap = (
            df_eval["anomaly_flow"].astype(bool) &
            df_eval["anomaly_simple"].astype(bool)
        ).sum()
        metrics["baseline_overlap"] = int(overlap)

    return metrics

In [ ]:
def compute_flow_metrics(df: pd.DataFrame):
    df_eval = df.dropna(subset=["anomaly_flow"])

    total = len(df_eval)
    anomalies = df_eval["anomaly_flow"].sum()
    rate = anomalies / total if total else 0

    metrics = {
        "total_points": total,
        "total_anomalies": int(anomalies),
        "anomaly_rate": float(rate),
        "score_mean": float(df_eval["flow_score"].mean()),
        "score_std": float(df_eval["flow_score"].std()),
        "score_p95": float(df_eval["flow_score"].quantile(0.95)),
        "score_p99": float(df_eval["flow_score"].quantile(0.99)),
        "score_max": float(df_eval["flow_score"].max()),
    }

    if "anomaly_simple" in df.columns:
        overlap = (
            df_eval["anomaly_flow"].astype(bool) &
            df_eval["anomaly_simple"].astype(bool)
        ).sum()
        metrics["baseline_overlap"] = int(overlap)

    return metrics

In [ ]:
def plot_flow_results(df, time_col="open_time", price_col="close"):
    d = df.dropna(subset=["anomaly_flow"])

    plt.figure()
    plt.plot(d[time_col], d[price_col])
    an = d[d["anomaly_flow"] == True]
    plt.scatter(an[time_col], an[price_col])
    plt.title("Precio con anomalías detectadas por Normalizing Flow")
    plt.show()

    plt.figure()
    plt.plot(d[time_col], d["flow_score"])
    plt.title("Normalizing Flow score en el tiempo")
    plt.show()

    plt.figure()
    plt.hist(d["flow_score"].dropna(), bins=50)
    plt.title("Distribución del Normalizing Flow score")
    plt.show()

In [ ]:
def plot_flow_pca_score(df, features, score_col="flow_score", title="Flow PCA score", cmap="RdYlGn_r", sample_n=30000, random_state=42):
    d = df.dropna(subset=features + [score_col]).copy()
    if len(d) > sample_n:
        d = d.sample(sample_n, random_state=random_state)

    X = d[features].replace([np.inf, -np.inf], np.nan).dropna()
    d = d.loc[X.index]
    scores = d[score_col].values

    Xs = StandardScaler().fit_transform(X.values)
    X2 = PCA(n_components=2, random_state=random_state).fit_transform(Xs)

    plt.figure()
    sc = plt.scatter(X2[:, 0], X2[:, 1], c=scores, cmap=cmap)
    plt.colorbar(sc, label="Anomaly Score")
    plt.title(title)
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.show()

In [ ]:
features = [
    "log_return",
    "volatility_20",
    "range_hl",
    "trades_per_volume",
    "buy_ratio"
]

In [ ]:
df_flow, flow_model, flow_scaler, flow_thr = fit_normalizing_flow(
    df=df,
    features=features,
    contamination=0.01,
    epochs=15,
    n_coupling=6,
    hidden_dim=128
)

In [ ]:
metrics_flow = compute_flow_metrics(df_flow)
print(metrics_flow)

In [ ]:
plot_flow_results(df_flow)
plot_flow_pca_score(df_flow, features, title="Normalizing Flow PCA score")